# Kata Unik


# Tugas 2

## 1. Install library terlebih dahulu

In [1]:
pip install pandas openpyxl numpy scikit-learn

## 2. Import library

In [2]:
import pandas as pd
import numpy as np
import re

from scipy.sparse import csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 3. Membaca dataset

In [3]:
df = pd.read_excel("dataset_detik_200_berita.xlsx")

df.head()

,id,isi_berita,label
0,1,Chef de Mission (CdM) Indonesia Todotua Pasari...,sport
1,2,"Banjir besar melanda Nagoya, Jepang, menjelang...",sport
2,3,Ketua Umum Komite Olimpiade Indonesia (KOI) Ra...,sport
3,4,Persaingan titel juara dunia semakin ketat men...,sport
4,5,Sektor ganda campuran kembali mengalami peromb...,sport


### Cek jumlah data:

In [4]:
print("Jumlah data :", len(df))
print("\nNama kolom:")
print(df.columns)

print("\nJumlah data per label:")
print(df["label"].value_counts())

Jumlah data : 200

Nama kolom:
Index(['id', 'isi_berita', 'label'], dtype='object')

Jumlah data per label:
label
sport      100
finance    100
Name: count, dtype: int64


## 4. Mengubah label menjadi numerik

In [5]:
df["label_num"] = df["label"].map({
    "sport": 1,
    "finance": 0
})

df[["id", "label", "label_num"]].head()

,id,label,label_num
0,1,sport,1
1,2,sport,1
2,3,sport,1
3,4,sport,1
4,5,sport,1


In [6]:
print(df["label_num"].value_counts())

label_num
1    100
0    100
Name: count, dtype: int64


## 5. Menghitung jumlah kata semua berita

In [7]:
df["jumlah_kata_asli"] = df["isi_berita"].astype(str).apply(
    lambda x: len(x.split())
)

df[["id", "jumlah_kata_asli"]].head()

total_kata = df["jumlah_kata_asli"].sum()

print("Total seluruh kata :", total_kata)

print(df["jumlah_kata_asli"].describe())

Total seluruh kata : 64635
count     200.000000
mean      323.175000
std       131.912543
min        41.000000
25%       243.000000
50%       299.500000
75%       365.250000
max      1037.000000
Name: jumlah_kata_asli, dtype: float64


### melihat jumlah kata setiap berita

In [8]:
df[[
    "id",
    "label",
    "jumlah_kata_asli"
]]

,id,label,jumlah_kata_asli
0,1,sport,300
1,2,sport,296
2,3,sport,402
3,4,sport,216
4,5,sport,189
...,...,...,...
195,196,finance,334
196,197,finance,493
197,198,finance,582
198,199,finance,196


## 6. Kamus kata tidak baku

In [9]:
kamus_tidak_baku = {
    "gak": "tidak",
    "nggak": "tidak",
    "ga": "tidak",
    "enggak": "tidak",
    "yg": "yang",
    "dgn": "dengan",
    "utk": "untuk",
    "krn": "karena",
    "kalo": "kalau",
    "kalok": "kalau",
    "aja": "saja",
    "udah": "sudah",
    "sdh": "sudah",
    "blm": "belum",
    "tdk": "tidak",
    "dr": "dari",
    "dlm": "dalam",
    "jd": "jadi",
    "bgt": "banget",
    "tp": "tetapi",
    "tapi": "tetapi",
    "karna": "karena",
    "trus": "terus",
    "kmrn": "kemarin",
    "dpt": "dapat",
    "hrs": "harus",
    "sm": "sama",
    "sy": "saya"
}

## 7. Kamus bahasa asing

In [10]:
kamus_asing = {

    # SPORT
    "rider": "pembalap",
    "race": "balapan",
    "racing": "balap",
    "team": "tim",
    "coach": "pelatih",
    "player": "pemain",
    "match": "pertandingan",
    "winner": "pemenang",
    "season": "musim",
    "training": "latihan",
    "game": "pertandingan",
    "games": "pertandingan",
    "manager": "manajer",
    "champion": "juara",
    "championship": "kejuaraan",
    "league": "liga",
    "score": "skor",
    "goal": "gol",
    "final": "final",

    # FINANCE
    "finance": "keuangan",
    "financial": "keuangan",
    "market": "pasar",
    "stock": "saham",
    "stocks": "saham",
    "sale": "penjualan",
    "price": "harga",
    "business": "bisnis",
    "company": "perusahaan",
    "investment": "investasi",
    "investor": "investor",
    "banking": "perbankan",
    "bank": "bank",
    "economy": "ekonomi",
    "economic": "ekonomi",
    "growth": "pertumbuhan",
    "profit": "keuntungan",
    "loss": "kerugian",
    "revenue": "pendapatan"
}

## 8. Fungsi preprocessing

* Mengubah Tipe Data (str(text)): Memastikan bahwa masukan bertipe data string (teks) untuk menghindari eror jika ada data kosong atau berupa angka.

* Case Folding (text.lower()): Mengubah semua huruf menjadi huruf kecil agar kata yang sama tidak dianggap berbeda hanya karena huruf kapital (misal: "Teks" dan "teks" dianggap sama).

* Hapus URL & Email: Menghilangkan tautan web (seperti http:// atau www.) dan alamat email menggunakan regex karena informasi ini biasanya tidak dibutuhkan dalam analisis teks.

* Hapus Angka: Menghilangkan semua karakter angka (0-9).

* Hapus Tanda Baca, Simbol, dan Emotikon: Menghapus semua karakter selain huruf alfabet dan karakter beraksen. Simbol, tanda baca, dan emoji otomatis diganti dengan spasi.

* Hapus Spasi Berlebih (.strip()): Merapikan spasi ganda akibat proses penghapusan sebelumnya menjadi satu spasi saja, sekaligus membuang spasi di awal dan akhir teks.

* Tokenisasi (text.split()): Memecah kalimat menjadi potongan kata-kata individu (token) berdasarkan spasi.

* Normalisasi Teks (Kamus Baku & Asing): Memeriksa setiap kata ke dalam kamus_tidak_baku untuk memperbaiki kata gaul/singkatan (misal: "yg" menjadi "yang"), dan memeriksa ke kamus_asing untuk menerjemahkan istilah asing ke bahasa Indonesia.

* Penggabungan Kembali (" ".join(hasil)): Menyatukan kembali potongan kata-kata yang sudah bersih dan baku tersebut menjadi satu kalimat utuh berspasi.




In [11]:
def preprocessing(text):

    # Pastikan berupa string
    text = str(text)

    # =========================
    # CASE FOLDING
    # =========================
    text = text.lower()


    # =========================
    # HAPUS URL
    # =========================
    text = re.sub(
        r'https?://\S+|www\.\S+',
        ' ',
        text
    )


    # =========================
    # HAPUS EMAIL
    # =========================
    text = re.sub(
        r'\S+@\S+',
        ' ',
        text
    )


    # =========================
    # HAPUS ANGKA
    # =========================
    text = re.sub(
        r'\d+',
        ' ',
        text
    )


    # =========================
    # HAPUS TANDA BACA,
    # SIMBOL DAN EMOTICON
    # =========================

    text = re.sub(
        r'[^a-zA-ZÀ-ÿ\s]',
        ' ',
        text
    )


    # =========================
    # HAPUS SPASI BERLEBIH
    # =========================

    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()


    # =========================
    # TOKENISASI
    # =========================

    kata = text.split()


    hasil = []

    for token in kata:

        # Membakukan kata
        if token in kamus_tidak_baku:
            token = kamus_tidak_baku[token]


        # Bahasa asing -> Indonesia
        if token in kamus_asing:
            token = kamus_asing[token]


        hasil.append(token)


    return " ".join(hasil)

## 9. Terapkan preprocessing ke semua berita

* Menerapkan Fungsi (.apply(preprocessing)): Menjalankan fungsi preprocessing (yang sudah dibahas sebelumnya) secara otomatis ke setiap baris data yang ada di dalam kolom "isi_berita".

* Membuat Kolom Baru (df["berita_clean"] = ...): Menyimpan seluruh hasil teks yang sudah dibersihkan tersebut ke dalam sebuah kolom baru bernama "berita_clean" di dalam tabel data (DataFrame).

* Memilih Kolom (df[["id", "isi_berita", "berita_clean"]]): Memilih hanya tiga kolom spesifik untuk ditampilkan, yaitu ID data, teks berita asli, dan teks berita yang sudah bersih.

* Menampilkan Cuplikan Data (.head()): Menampilkan 5 baris pertama dari tabel tersebut sehingga Anda bisa mengecek dan membandingkan langsung perbedaan antara teks sebelum dan sesudah dibersihkan.

In [12]:
df["berita_clean"] = df["isi_berita"].apply(preprocessing)

df[[
    "id",
    "isi_berita",
    "berita_clean"
]].head()

,id,isi_berita,berita_clean
0,1,Chef de Mission (CdM) Indonesia Todotua Pasari...,chef de mission cdm indonesia todotua pasaribu...
1,2,"Banjir besar melanda Nagoya, Jepang, menjelang...",banjir besar melanda nagoya jepang menjelang a...
2,3,Ketua Umum Komite Olimpiade Indonesia (KOI) Ra...,ketua umum komite olimpiade indonesia koi raja...
3,4,Persaingan titel juara dunia semakin ketat men...,persaingan titel juara dunia semakin ketat men...
4,5,Sektor ganda campuran kembali mengalami peromb...,sektor ganda campuran kembali mengalami peromb...


## 10. Hitung jumlah kata setelah preprocessing

Kode ini berfungsi untuk menghitung total kata pada teks berita yang sudah dibersihkan, lalu menyimpan hasilnya ke dalam kolom baru bernama jumlah_kata_clean. Setelah itu, kode akan menampilkan cuplikan 5 baris pertama untuk membandingkan jumlah kata sebelum dan sesudah proses pembersihan.

In [13]:
df["jumlah_kata_clean"] = df["berita_clean"].apply(
    lambda x: len(x.split())
)

df[[
    "id",
    "jumlah_kata_asli",
    "jumlah_kata_clean"
]].head()

,id,jumlah_kata_asli,jumlah_kata_clean
0,1,300,295
1,2,296,290
2,3,402,399
3,4,216,214
4,5,189,193


### Total:

In [14]:
print(
    "Total kata sebelum preprocessing :",
    df["jumlah_kata_asli"].sum()
)

print(
    "Total kata setelah preprocessing :",
    df["jumlah_kata_clean"].sum()
)

Total kata sebelum preprocessing : 64635
Total kata setelah preprocessing : 63542


## 11. Ekstrak seluruh kata unik

Kode ini berfungsi untuk mengumpulkan dan menghitung seluruh kata dari semua teks berita yang sudah dibersihkan.

In [15]:
semua_kata = []

for berita in df["berita_clean"]:
    semua_kata.extend(
        berita.split()
    )

kata_unik = sorted(
    set(semua_kata)
)

print(
    "Jumlah seluruh kata:",
    len(semua_kata)
)

print(
    "Jumlah kata unik:",
    len(kata_unik)
)

Jumlah seluruh kata: 63542
Jumlah kata unik: 7343


In [16]:
kata_unik[:100]

['a',
 'aam',
 'aan',
 'aau',
 'abadi',
 'abal',
 'abang',
 'abdi',
 'abdul',
 'aberdeen',
 'abraham',
 'absen',
 'absennya',
 'abu',
 'abullah',
 'acara',
 'accurate',
 'acd',
 'aceh',
 'achmad',
 'acid',
 'acosta',
 'activ',
 'activation',
 'activities',
 'acuan',
 'acuannya',
 'ada',
 'adalah',
 'adanya',
 'adaptasi',
 'adapun',
 'adhitya',
 'adianto',
 'adik',
 'adil',
 'adininggar',
 'adk',
 'administrasi',
 'administratif',
 'adna',
 'adopsi',
 'adp',
 'adriatik',
 'adu',
 'aduan',
 'advokat',
 'aerodinamika',
 'aeromodelling',
 'aesi',
 'af',
 'aff',
 'afrianto',
 'afrika',
 'ag',
 'agak',
 'agama',
 'agar',
 'agency',
 'agenda',
 'agent',
 'agraria',
 'agreement',
 'agresif',
 'agresivitas',
 'agung',
 'agus',
 'agustus',
 'agustusan',
 'ahi',
 'ahlinya',
 'ahmad',
 'ahmed',
 'ahren',
 'ai',
 'aichi',
 'air',
 'airin',
 'airlangga',
 'airport',
 'ajaib',
 'ajakan',
 'ajaknya',
 'ajang',
 'ajaran',
 'aji',
 'akademi',
 'akademik',
 'akademisi',
 'akal',
 'akan',
 'akbar',
 'akd'

In [17]:
df_kata_unik = pd.DataFrame({
    "kata_unik": kata_unik
})

df_kata_unik.head(20)

,kata_unik
0,a
1,aam
2,aan
3,aau
4,abadi
5,abal
6,abang
7,abdi
8,abdul
9,aberdeen


In [18]:
df_kata_unik.to_excel(
    "kata_unik.xlsx",
    index=False
)

## 12. Membagi data 160 training dan 40 testing

Kode ini berfungsi untuk membagi dataset menjadi dua bagian, yaitu data latih (train) dan data uji (test) yang nantinya akan digunakan untuk melatih dan menguji model machine learning.

In [19]:
X_train_text, X_test_text, y_train, y_test = train_test_split(

    df["berita_clean"],
    df["label_num"],

    test_size=40,

    random_state=42,

    stratify=df["label_num"]
)

In [20]:
print(
    "Jumlah training:",
    len(X_train_text)
)

print(
    "Jumlah testing:",
    len(X_test_text)
)

Jumlah training: 160
Jumlah testing: 40


In [21]:
print("TRAINING")
print(y_train.value_counts())

print("\nTESTING")
print(y_test.value_counts())

TRAINING
label_num
0    80
1    80
Name: count, dtype: int64

TESTING
label_num
0    20
1    20
Name: count, dtype: int64


## 13. Representasi TF-IDF

Kode ini berfungsi untuk mengubah data teks berita menjadi format angka (vektor numerik) menggunakan metode TF-IDF (Term Frequency-Inverse Document Frequency) agar bisa dipahami dan diproses oleh algoritma machine learning.

Selain mengubah teks menjadi angka berdasarkan bobot kepentingan kata, kode ini juga secara otomatis menyaring kosa kata, yaitu membuang kata yang terlalu jarang muncul (muncul di kurang dari 2 berita) dan kata yang terlalu sering muncul (muncul di lebih dari 95% berita). Pola kosa kata ini dipelajari dari data latih (train), lalu diterapkan untuk mengubah data latih dan data uji (test) ke dalam bentuk angka.

In [22]:
tfidf = TfidfVectorizer(
    min_df=2,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(
    X_train_text
)

X_test_tfidf = tfidf.transform(
    X_test_text
)

In [23]:
nama_fitur = tfidf.get_feature_names_out()

print(
    "Jumlah fitur TF-IDF:",
    len(nama_fitur)
)

Jumlah fitur TF-IDF: 3159


## 14. Melihat matriks TF-IDF

Kode ini berfungsi untuk mengubah hasil perhitungan TF-IDF (yang sebelumnya berupa format matriks matematika) menjadi sebuah tabel (DataFrame) yang rapi dan mudah dibaca.

Dalam tabel ini, setiap kolom mewakili satu kosa kata yang berhasil diekstrak dari berita, dan angka-angka di dalamnya menunjukkan nilai bobot kepentingan kata tersebut pada setiap berita. Setelah tabelnya dibuat, kode akan menampilkan cuplikan 5 baris pertamanya agar Anda bisa melihat langsung hasil dari pembobotan kata-kata tersebut.

In [24]:
tfidf_train_df = pd.DataFrame(
    X_train_tfidf.toarray(),
    columns=nama_fitur
)

tfidf_train_df.head()

,abdul,abu,acara,acd,aceh,acosta,acuan,ada,adalah,adanya,...,yogyakarta,youtube,yudhi,yusrian,yusuf,zapp,zona,zoom,zulhas,zulkifli
0,0.0,0.097117,0.0,0.094258,0.0,0.0,0.0,0.021731,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.078395,0.0,0.049221,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [25]:
tfidf_train_df["label"] = y_train.reset_index(
    drop=True
)

tfidf_train_df.head()

,abdul,abu,acara,acd,aceh,acosta,acuan,ada,adalah,adanya,...,youtube,yudhi,yusrian,yusuf,zapp,zona,zoom,zulhas,zulkifli,label
0,0.0,0.097117,0.0,0.094258,0.0,0.0,0.0,0.021731,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.078395,0.0,0.049221,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


## 15. Simpan data TF-IDF

Kode ini berfungsi untuk **menyimpan tabel hasil perhitungan TF-IDF yang sudah dibuat tadi ke dalam sebuah file Excel** bernama `"data_tfidf_training.xlsx"`.

Pengaturan `index=False` digunakan untuk memastikan bahwa nomor urut baris bawaan dari sistem tidak ikut dimasukkan ke dalam file Excel tersebut, sehingga hasil akhirnya menjadi lebih rapi dan hanya berisi nama kolom/kata beserta nilai bobotnya saja.

In [26]:
tfidf_train_df.to_excel(
    "data_tfidf_training.xlsx",
    index=False
)

In [27]:
tfidf_test_df = pd.DataFrame(
    X_test_tfidf.toarray(),
    columns=nama_fitur
)

tfidf_test_df["label"] = y_test.reset_index(
    drop=True
)

tfidf_test_df.to_excel(
    "data_tfidf_testing.xlsx",
    index=False
)

In [28]:
features = np.array(
    tfidf.get_feature_names_out()
)

jumlah_kelas = y_train.nunique()

class_space_density = np.zeros(
    len(features)
)

y_train_array = y_train.to_numpy()

In [29]:
for kelas in sorted(
    y_train.unique()
):

    mask = (
        y_train_array == kelas
    )

    X_class = X_train_tfidf[
        mask
    ]


    # Berapa dokumen kelas tersebut
    # mengandung sebuah kata

    document_frequency_class = (
        (X_class > 0)
        .sum(axis=0)
        .A1
    )


    jumlah_dokumen_class = (
        X_class.shape[0]
    )


    class_density = (
        document_frequency_class
        /
        jumlah_dokumen_class
    )


    class_space_density += (
        class_density
    )

In [30]:
epsilon = 1e-12

icsdf = np.log(

    (jumlah_kelas + epsilon)

    /

    (class_space_density + epsilon)
)

In [31]:
df_icsdf = pd.DataFrame({

    "kata": features,

    "ICSDF": icsdf
})

df_icsdf.sort_values(
    "ICSDF",
    ascending=False
).head(20)

,kata,ICSDF
3158,zulkifli,4.382027
3157,zulhas,4.382027
3156,zoom,4.382027
3137,wujud,4.382027
3136,wrt,4.382027
3133,wong,4.382027
3130,wisatawan,4.382027
3129,wiryawan,4.382027
3128,wild,4.382027
3125,whoosh,4.382027


## 17. TF-IDF × ICSDF

In [32]:
X_train_icsdf = X_train_tfidf.multiply(
    icsdf
)

X_test_icsdf = X_test_tfidf.multiply(
    icsdf
)

In [33]:
X_train_icsdf = csr_matrix(
    X_train_icsdf
)

X_test_icsdf = csr_matrix(
    X_test_icsdf
)

## 18. Menentukan kata paling penting

Kode ini berfungsi untuk menghitung rata-rata nilai bobot setiap kata (fitur) di seluruh data latih.

Hasil perhitungan rata-rata tersebut kemudian diubah bentuknya dan diratakan menjadi sebuah daftar angka satu dimensi (seperti sebuah baris panjang berisi nilai-nilai). Hal ini biasanya dilakukan agar skor dari tiap kata tersebut lebih mudah diproses pada tahap selanjutnya, misalnya untuk mengurutkan atau menyeleksi kata mana yang memiliki skor rata-rata paling tinggi.

In [34]:
skor_fitur = np.asarray(
    X_train_icsdf.mean(
        axis=0
    )
).ravel()

In [35]:
TOP_K = 100

In [36]:
top_index = np.argsort(
    skor_fitur
)[::-1][:TOP_K]

In [37]:
kata_penting = features[
    top_index
]

kata_penting[:30]

array(['saham', 'motogp', 'padel', 'purbaya', 'marc', 'set', 'marquez',
       'balapan', 'harga', 'asian', 'reza', 'martin', 'pertandingan',
       'rp', 'sabar', 'olahraga', 'bandara', 'pajak', 'emas', 'saya',
       'gram', 'aragon', 'atlet', 'morgan', 'antonelli', 'energi',
       'bantuan', 'bbm', 'ethan', 'pertashop'], dtype=object)

In [38]:
df_kata_penting = pd.DataFrame({

    "kata": kata_penting,

    "skor": skor_fitur[
        top_index
    ]
})

df_kata_penting.head(30)

,kata,skor
0,saham,0.053538
1,motogp,0.049970
2,padel,0.048358
3,purbaya,0.047914
4,marc,0.047852
5,set,0.047083
6,marquez,0.046072
7,balapan,0.043912
8,harga,0.040216
9,asian,0.039896


## 19. Reduksi menjadi 100 fitur ICSDF

Kode ini berfungsi untuk memfilter atau menyeleksi kata-kata (fitur) yang paling penting untuk dimasukkan ke dalam model machine learning.

In [39]:
X_train_selected = X_train_icsdf[
    :,
    top_index
]

X_test_selected = X_test_icsdf[
    :,
    top_index
]

In [40]:
print(
    "Sebelum seleksi:",
    X_train_tfidf.shape
)

print(
    "Sesudah ICSDF:",
    X_train_selected.shape
)

Sebelum seleksi: (160, 3159)
Sesudah ICSDF: (160, 100)


## 20. Tabel hasil ICSDF

Kode ini berfungsi untuk mengubah data kata-kata terpilih menjadi sebuah tabel (DataFrame) yang rapi, sekaligus menyatukannya dengan label kategori berita tersebut.

In [41]:
icsdf_train_df = pd.DataFrame(

    X_train_selected.toarray(),

    columns=kata_penting
)

icsdf_train_df["label"] = (

    y_train.reset_index(
        drop=True
    )
)

icsdf_train_df.head()

,saham,motogp,padel,purbaya,marc,set,marquez,balapan,harga,asian,...,seri,kota,subsidi,ekonomi,inggris,jepang,bisnis,petenis,bulog,label
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.478319,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.000000,0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.000000,1
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.909254,0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.436518,...,0.0,0.400006,0.0,0.0,0.0,0.31316,0.0,0.0,0.000000,1
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.000000,1


In [42]:
icsdf_train_df.to_excel(
    "data_icsdf_training.xlsx",
    index=False
)

## 21. PCA

Kode ini berfungsi untuk mempersiapkan metode PCA (Principal Component Analysis) untuk menyederhanakan atau mengurangi jumlah fitur pada data.

In [43]:
pca = PCA(

    n_components=20,

    random_state=42
)

In [44]:
X_train_selected_dense = (
    X_train_selected.toarray()
)

X_test_selected_dense = (
    X_test_selected.toarray()
)

In [45]:
X_train_pca = pca.fit_transform(
    X_train_selected_dense
)

X_test_pca = pca.transform(
    X_test_selected_dense
)

In [46]:
print(
    X_train_pca.shape
)

print(
    X_test_pca.shape
)

(160, 20)
(40, 20)


## 22. Membuat tabel data reduksi training

Kode ini berfungsi untuk membuat daftar nama secara otomatis yang berurut mulai dari "PC1" hingga "PC20".

In [47]:
nama_pc = [
    f"PC{i}"
    for i in range(
        1,
        21
    )
]

In [48]:
df_train_reduksi = pd.DataFrame(

    X_train_pca,

    columns=nama_pc
)

df_train_reduksi["label"] = (

    y_train.reset_index(
        drop=True
    )
)

df_train_reduksi.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,label
0,0.125175,0.042469,0.040138,-0.019667,-0.034541,-0.031756,-0.022559,0.066880,-0.004115,-0.009071,...,-0.011961,-0.057276,0.005319,-0.033165,0.064557,0.006048,-0.011198,-0.020397,-0.002022,0
1,-0.435953,0.462965,-0.021210,-0.035918,-0.080061,0.217714,-0.031991,-0.015585,-0.008987,0.001953,...,0.012256,-0.375203,-1.578091,1.409892,0.116971,-0.320665,0.276590,0.187618,0.026366,1
2,0.020279,-0.123370,0.158263,-0.410967,-0.573201,-1.086913,-1.166716,-1.338402,0.689620,-0.215426,...,0.014522,0.340521,0.018416,0.088827,0.013663,-0.080783,0.066621,0.051994,0.018552,0
3,-0.010471,-0.005318,0.037714,-0.123128,0.003710,0.011467,-0.012336,0.016158,-0.027880,0.001557,...,0.009293,-0.148085,-0.068652,-0.044826,-0.038476,0.223258,-0.209596,-0.223745,-0.016296,1
4,-0.428090,0.492397,0.003432,-0.036958,-0.008599,0.166387,-0.010077,-0.038749,0.024427,-0.001769,...,-0.000059,-0.072038,-0.341839,0.298968,0.020498,-0.013082,0.011358,-0.014596,-0.005602,1


## 23. Data testing

Kode ini berfungsi untuk mengubah data uji (test) yang sudah disederhanakan oleh PCA menjadi sebuah tabel yang rapi beserta label kategorinya.

In [49]:
df_test_reduksi = pd.DataFrame(

    X_test_pca,

    columns=nama_pc
)

df_test_reduksi["label"] = (

    y_test.reset_index(
        drop=True
    )
)

df_test_reduksi.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,label
0,0.064907,-0.013507,0.093718,-0.071965,-0.053244,-0.063072,-0.037004,0.127958,-0.107261,-0.025465,...,-0.008686,-0.201978,0.083923,-0.019586,-0.016270,-0.008431,-0.027480,-0.039435,-0.111958,0
1,-0.029207,-0.068524,-0.039877,-0.025922,0.000697,-0.020597,0.033532,0.008385,-0.027807,0.042570,...,0.050764,-0.074896,-0.031897,-0.026400,-0.018879,0.085956,-0.097974,-0.111319,0.000022,1
2,0.004381,-0.023716,0.034213,-0.094245,0.030332,-0.017835,-0.002313,0.026456,-0.034078,0.002662,...,0.007229,-0.103073,-0.026033,-0.045856,-0.025888,0.122502,-0.118622,-0.115787,-0.020131,1
3,0.000291,-0.025887,0.033054,-0.080641,0.017762,-0.018608,-0.004718,0.025488,-0.026129,0.003143,...,0.006173,-0.086954,-0.019613,-0.037408,-0.018899,0.094870,-0.090485,-0.078215,-0.020582,1
4,-0.003314,0.005919,0.050265,-0.053421,0.015753,-0.028732,-0.019515,0.047576,-0.042212,-0.011945,...,-0.005354,-0.113131,-0.018144,0.002356,0.004407,0.080025,-0.082166,0.074540,-0.104181,0


## 24. Simpan data reduksi

Kode ini berfungsi untuk menyimpan dua tabel data (data latih dan data uji) yang sudah disederhanakan jumlah fiturnya tadi ke dalam bentuk dua file Excel terpisah.

In [50]:
df_train_reduksi.to_excel(
    "data_reduksi_training.xlsx",
    index=False
)

df_test_reduksi.to_excel(
    "data_reduksi_testing.xlsx",
    index=False
)

## 25. Cek total training dan testing

Kode ini berfungsi untuk menampilkan ukuran atau dimensi (jumlah baris dan jumlah kolom) dari tabel data latih (training) dan data uji (testing) yang telah disederhanakan sebelumnya.

In [51]:
print(
    "Training :",
    df_train_reduksi.shape
)

print(
    "Testing :",
    df_test_reduksi.shape
)

Training : (160, 21)
Testing : (40, 21)


## 26. Kolom terakhir harus label

In [52]:
df_train_reduksi.columns

Index(['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10',
       'PC11', 'PC12', 'PC13', 'PC14', 'PC15', 'PC16', 'PC17', 'PC18', 'PC19',
       'PC20', 'label'],
      dtype='object')